In [143]:
from langchain_core.messages import HumanMessage,SystemMessage
from langgraph.graph import StateGraph,START, END
from typing import TypedDict , Literal, Annotated
from langchain_groq import ChatGroq
from pydantic import BaseModel,Field
import operator
from dotenv import load_dotenv
load_dotenv()

True

In [144]:
evaluate_model=ChatGroq(
    model = "llama-3.3-70b-versatile"
)

generate_post=ChatGroq(
    model = "llama-3.3-70b-versatile"
)

optimize_post=ChatGroq(
    model = "llama-3.3-70b-versatile"
)

In [145]:
class x_state(TypedDict):
    topic:str
    tweet:str
    evaluation:Literal["approved","needs_improvement"]
    iteration:int
    max_iteration:int
    feedback:str
    tweet_history:Annotated[list[str],operator.add]
    feedback_history:Annotated[list[str],operator.add]

In [146]:
class EvaluateStructure(BaseModel):
    feedback: str = Field(description="Provide feedback on the given tweet/post")
    evaluation: Literal["approved", "needs_improvement"] = Field(
        description="Final decision after evaluation"
    )

In [147]:
def generate_tweet(state:x_state):
    messages = [
    SystemMessage(content="You are a funny and clever Twitter/X influencer."),
    HumanMessage(content=f"""
    Write a short, original, and hilarious tweet on the topic: "{state['topic']}".

    Rules:
    - Do NOT use question-answer format.
    - Max 280 characters.
    - Use observational humor, irony, sarcasm, or cultural references.
    - Think in meme logic, punchlines, or relatable takes.
    - Use simple, day to day english
    - This is version {state['iteration'] + 1}.
    """)
    ]
    tweet=generate_post.invoke(messages).content

    return {"tweet":tweet,"tweet_history":[tweet]}

In [148]:
def evaluate_tweet(state:x_state):
    messages = [
    SystemMessage(content="You are a ruthless, no-laugh-given Twitter critic. You evaluate tweets based on humor, originality, virality, and tweet format."),
    HumanMessage(content=f"""
    Evaluate the following tweet:

    Tweet: "{state['tweet']}"

    Use the criteria below to evaluate the tweet:

    1. Originality – Is this fresh, or have you seen it a hundred times before?  
    2. Humor – Did it genuinely make you smile, laugh, or chuckle?  
    3. Punchiness – Is it short, sharp, and scroll-stopping?  
    4. Virality Potential – Would people retweet or share it?  
    5. Format – Is it a well-formed tweet (not a setup-punchline joke, not a Q&A joke, and under 280 characters)?

    Auto-reject if:
    - It's written in question-answer format (e.g., "Why did..." or "What happens when...")
    - It exceeds 280 characters
    - It reads like a traditional setup-punchline joke
    - Dont end with generic, throwaway, or deflating lines that weaken the humor (e.g., “Masterpieces of the auntie-uncle universe” or vague summaries)

    ### Respond ONLY in structured format:
    - evaluation: "approved" or "needs_improvement"  
    - feedback: One paragraph explaining the strengths and weaknesses 
    """)
    ]
    evaluate=evaluate_model.with_structured_output(EvaluateStructure)
    output=evaluate.invoke(messages)
    return {"feedback":output.feedback, "evaluation":[output.evaluation],"feedback_history":[output.feedback]}



In [149]:
def optimize_tweet(state:x_state):
    message=[
    SystemMessage(content="You punch up tweets for virality and humor based on given feedback."),
    HumanMessage(content=f"""
    Improve the tweet based on this feedback:
    "{state['feedback']}"

    Topic: "{state['topic']}"
    Original Tweet:
    {state['tweet']}

    Re-write it as a viral-worthy post. Avoid Q&A format for better readability.
    """)
    ]
    response=optimize_post.invoke(message).content
    iteration=state['iteration'] +1
    return {"tweet":response, "iteration":iteration,"tweet_history":[response]}   

In [150]:
def route_evaluation(state:x_state):
    if state['evaluation']=="approved" or state['iteration'] >=state['max_iteration']:
        return "approved" 
    else:
        return "needs_improvement"
    

In [151]:
graph=StateGraph(x_state)
graph.add_node("generate_tweet",generate_tweet)
graph.add_node("evaluate_tweet",evaluate_tweet)
graph.add_node("optimize_tweet",optimize_tweet)

graph.add_edge(START,"generate_tweet")
graph.add_edge("generate_tweet","evaluate_tweet")
graph.add_conditional_edges("evaluate_tweet",route_evaluation,{"approved":END,"needs_improvement":"optimize_tweet"})
graph.add_edge("optimize_tweet","evaluate_tweet")
workflow=graph.compile()

In [152]:
intial_state={
    "topic":"pakistan railway",
    "iteration":1,
    "max_iteration":5,
              }
workflow.invoke(intial_state)

{'topic': 'pakistan railway',
 'tweet': '"Pakistan Railway: where the journey is crazier than the destination. Think spontaneous dance parties in the aisles, mysterious stops in the middle of nowhere, and food so good you\'ll forget where you\'re going. All aboard for the most unpredictable, hilarious, and deliciously chaotic train ride of your life #PakistanRailwayShenanigans"',
 'evaluation': ['approved'],
 'iteration': 5,
 'max_iteration': 5,
 'feedback': "This tweet is a breath of fresh air, showcasing originality and humor. The use of vivid descriptions like 'spontaneous dance parties in the aisles' and 'food so good you'll forget where you're going' effectively paint a picture of the chaotic yet exciting experience of Pakistan Railway. The tweet's punchiness is also notable, as it's concise and scroll-stopping. The virality potential is high, given the unique blend of humor and cultural insight. The format is well-executed, avoiding common pitfalls like question-answer format and